# Feature extraction — CHNMF (Correntropy-based Hypergraph regularized NMF)

Theo Yu, Wu, Liu, Zheng & Xu, *Correntropy-Based Hypergraph Regularized NMF for Clustering and Feature Selection on Multi-Cancer Integrated Data*, IEEE Trans. Cybernetics 51(8) (2021) 3952–3963.

Ký hiệu như bài báo: **X ∈ ℝ^{m×n}** (m gene là hàng, n mẫu là cột — notebook chuyển vị ma trận mẫu × gene), **X ≈ UVᵀ**, **U ∈ ℝ^{m×k}** (ma trận cơ sở / metagene), **V ∈ ℝ^{n×k}** (ma trận hệ số của mẫu). **Đặc trưng mới của mỗi mẫu là hàng tương ứng của V** (bài báo phân cụm trên V và chọn gene trên U).

- **Hàm mục tiêu (12):** max Σᵢ g(‖xⁱ − (UVᵀ)ⁱ‖) − λ·Tr(VᵀL_hyper V), với g là kernel Gauss (correntropy, tính theo **từng hàng = từng gene**) thay cho chuẩn Euclid ⇒ bền vững với nhiễu/ngoại lai.
- **Half-quadratic (13)–(20):** biến bài toán thành NMF có trọng số min Tr((X − UVᵀ)ᵀD(X − UVᵀ)) + λ·Tr(VᵀL_hyper V), với **D_ii = exp(−eᵢ / 2σ²)**, eᵢ = Σⱼ (X − UVᵀ)ᵢⱼ² và băng thông **σ² = (1/m)·Σᵢ eᵢ** (16).
- **Hypergraph (7)–(11):** mỗi mẫu vᵢ sinh 1 siêu cạnh eᵢ = {vᵢ} ∪ kNN(vᵢ); trọng số Wᵢ = Σ_{vⱼ∈eᵢ} exp(−‖vᵢ − vⱼ‖² / δ), δ = Σ_{vⱼ∈eᵢ} ‖vᵢ − vⱼ‖² / k; d(v) = Σₑ w(e)H(v,e), f(e) = |e|; **L_hyper = D_v − H W D_e⁻¹ Hᵀ**.
- **Algorithm 1** — lặp: cập nhật σ (16) → D (20) → U (26): `U ← U ⊙ (DXV) / (DUVᵀV)` → V (27): `V ← V ⊙ (XᵀDU + λ·E V) / (VUᵀDU + λ·D_v V)`. Ở (27), L_hyper = D_v − E được tách thành phần dương/âm (như GNMF/HNMF) để V luôn không âm. Dừng khi thay đổi tương đối của hàm mục tiêu (18) < `CHNMF_TOL` hoặc đạt `CHNMF_MAX_ITER` (bài báo: hội tụ trong ~100 vòng).

**Điều chỉnh cho bài toán phân loại (fit trên train, biến đổi val/test):**
- Bài báo chỉ phân rã một ma trận cố định. Ở đây U, D (trọng số correntropy từng gene) và L_hyper được học **chỉ trên train**. Mẫu val/test không thuộc hypergraph nên V_new được giải với **U, D cố định** bằng cùng luật nhân (27) bỏ số hạng hypergraph: `V ← V ⊙ (XᵀDU) / (VUᵀDU)` (`CHNMF_TRANSFORM_ITER` vòng).
- **λ:** bài báo chọn λ ∈ {10⁻⁵,…,10⁵} bằng 5-fold CV theo chất lượng phân cụm (giá trị tốt nhất 10⁻¹–10²). Mặc định `CHNMF_LAMBDA = 1`. Đặt `CHNMF_LAMBDA_GRID = [...]` để notebook tự chọn λ trên **train** của fold theo độ chính xác phân cụm AC (K-means trên V, ánh xạ nhãn bằng Kuhn–Munkres, như mục IV-D) — chi phí × số giá trị λ.
- Bài báo không nêu số láng giềng k của hypergraph → mặc định `CHNMF_HG_K = 5` (như HNMF, Zeng et al. 2014). Bài báo PCA dữ liệu xuống 2000 chiều trước; ở đây phân rã trực tiếp ma trận gene (NMF cần dữ liệu không âm, còn PC có giá trị âm).
- NMF cần **dữ liệu không âm**: nếu train có giá trị âm, tịnh tiến bởi `−min(train)` (hằng số tính trên train, giá trị âm còn lại ở val/test cắt về 0). Không chuẩn hoá.
- Ngoài đặc trưng V, notebook lưu **điểm gene** theo U (phần feature selection của bài báo): score(gene i) = Σⱼ uᵢⱼ (U chuẩn hoá cột), top `N_TOP_SCORED_GENES` (bài báo lấy top 500) vào `gene_scores/fold_k.csv`.

- **Khác với feature selection:** đặc trưng mới là **tổ hợp của gene** (không còn là gene gốc). Mỗi fold: **fit chỉ trên `train`**, sau đó dùng mô hình đã fit để biến đổi `val` và `test` (val/test không tham gia fit). Dữ liệu đầu vào giữ nguyên log2 expression, không chuẩn hoá (trừ khi ghi rõ bên dưới).
- **Đầu vào:** `DATA_DIR/fold_{1..5}/{train,val,test}.h5`.
- **Đổi dataset:** sửa `DATA_DIR` ở cell cấu hình. Số chiều đầu ra: `N_FEATURES` (mặc định 100).
- **Đầu ra** (`/kaggle/working/feature_extraction/chnmf/<dataset>/`):
  - `features/fold_k/{train,val,test}.h5` — dữ liệu sau trích xuất, **cùng format với dữ liệu gốc** (pandas, key=`"data"`, index = sample_id, cột = `CHNMF1..CHNMFk` + `label`). Notebook phân loại chỉ cần trỏ `DATA_DIR`/`DATA_ROOT` vào thư mục `features/`.
  - `components/fold_k.csv`, `components_all_folds.csv` — thông tin từng thành phần (rank, score: tỉ lệ đóng góp của thành phần ‖u_j‖·‖v_j‖ (chuẩn hoá tổng = 1, xếp giảm dần))
  - `loadings_top_genes.csv` — top `N_TOP_GENES` gene có trọng số |loading| lớn nhất của mỗi thành phần (diễn giải sinh học)
  - `metrics_per_fold.csv`, `metrics_summary.csv` — thời gian, CPU utilization, peak memory, energy, EDP, carbon footprint
- **Đo lường:** `ResourceTracker` chỉ bao quanh lời gọi `fit_extractor` (fit trên train + biến đổi train), không tính thời gian đọc file. Thời gian biến đổi val+test ghi riêng ở cột `transform_time_sec`. Energy/Carbon ước lượng bằng `codecarbon` (tracking_mode="process"); EDP = Energy(J) × Time(s).

In [ ]:
# tables: backend cho pd.read_hdf | codecarbon: do nang luong/CO2 | psutil: CPU & RAM
get_ipython().system("pip install -q codecarbon psutil tables")

In [ ]:
import os

# ====== CAU HINH (sua tay) ======
# Doi DATA_DIR sang bo khac (vd .../h5/tcga_luad) de chay dataset khac
DATA_DIR = "/kaggle/input/datasets/lfreedom2750/experiment-dataset/h5/gse68465"
N_FEATURES = 100          # so chieu dac trung moi sau trich xuat
N_FOLDS = 5
SEED = 42
METHOD = "chnmf"
CHNMF_LAMBDA = 1.0            # lambda: trong so hypergraph regularization
CHNMF_LAMBDA_GRID = None      # None | vd [1e-2, 1e-1, 1, 1e1, 1e2]: chon lambda tren train theo AC phan cum
CHNMF_HG_K = 5                # so lang gieng gan nhat khi tao sieu canh
CHNMF_MAX_ITER = 500
CHNMF_TOL = 1e-5              # dung khi thay doi tuong doi cua ham muc tieu (18) < tol
CHNMF_TRANSFORM_ITER = 300    # so vong giai V cho mau val/test (U, D co dinh)
N_TOP_SCORED_GENES = 500      # so gene luu trong gene_scores/ (bai bao: top 500)
METHOD_PARAMS = {"lambda": CHNMF_LAMBDA, "lambda_grid": CHNMF_LAMBDA_GRID, "hypergraph_k": CHNMF_HG_K, "max_iter": CHNMF_MAX_ITER, "tol": CHNMF_TOL, "transform_iter": CHNMF_TRANSFORM_ITER, "init": "random", "reference": "Yu et al., IEEE TCYB 51(8) 2021 3952-3963"}
# None -> codecarbon tu dinh vi (can Internet); hoac dat ma ISO 3 ky tu, vd "USA", "VNM"
COUNTRY_ISO_CODE = None

DATASET_NAME = os.path.basename(os.path.normpath(DATA_DIR))
RESULT_DIR = os.path.join("/kaggle/working/feature_extraction", METHOD, DATASET_NAME)
os.makedirs(os.path.join(RESULT_DIR, "components"), exist_ok=True)

for _s in ["train", "val", "test"]:
    assert os.path.isfile(os.path.join(DATA_DIR, "fold_1", f"{_s}.h5")), \
        f"Khong thay {DATA_DIR}/fold_1/{_s}.h5 -- da Add Input dataset chua?"
print("Dataset :", DATASET_NAME, "->", DATA_DIR)
print("Method  :", METHOD, "| N_FEATURES =", N_FEATURES)
print("Ket qua :", RESULT_DIR)
print("So CPU  :", os.cpu_count())

## Đọc dữ liệu

In [ ]:
import json, time, threading, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

STAGE_NAMES = {0: "Normal", 1: "I", 2: "II", 3: "III", 4: "IV"}


def load_split(fold, split="train"):
    # File h5 (pandas, key="data"): index = sample_id, cot = ten gene + cot "label"
    df = pd.read_hdf(os.path.join(DATA_DIR, f"fold_{fold}", f"{split}.h5"), key="data")
    X = df.drop(columns="label")
    # Giu nguyen gia tri log2 expression, KHONG chuan hoa
    return X.values.astype(np.float64), df["label"].values.astype(int), X.columns.to_numpy(), df.index.to_numpy()


# Doc thu fold 1 de xem cau truc
X0, y0, genes0, ids0 = load_split(1)
print(f"fold_1/train: {X0.shape[0]} mau x {X0.shape[1]} gene | khoang gia tri [{X0.min():.2f}, {X0.max():.2f}]")
print("Vi du gene:", list(genes0[:5]), "| vi du sample:", list(ids0[:3]))
print("Phan bo lop:", pd.Series(y0).map(STAGE_NAMES).value_counts().to_dict())
del X0, y0, genes0, ids0

## Bộ đo tài nguyên

In [ ]:
import psutil

try:
    from codecarbon import EmissionsTracker, OfflineEmissionsTracker
    _CODECARBON = True
except ImportError:
    _CODECARBON = False


class ResourceTracker:
    """Do tren 1 khoi lenh: thoi gian (s), CPU utilization, peak memory (RSS, gom ca
    tien trinh con cua joblib), energy (kWh, codecarbon), carbon (kg CO2eq) va
    Energy-Delay Product EDP = Energy(J) * Time(s)."""

    def __init__(self, name, interval=0.05):
        self.name, self.interval = name, interval
        self.proc = psutil.Process()
        self._stop = threading.Event()

    def _procs(self):
        try:
            return [self.proc] + self.proc.children(recursive=True)
        except psutil.Error:
            return [self.proc]

    def _poll(self):
        # RSS tong va CPU time cua worker con (tru phan CPU worker da dung truoc khi bat dau do)
        rss = 0
        for p in self._procs():
            try:
                rss += p.memory_info().rss
                if p.pid != self.proc.pid:
                    t = p.cpu_times()
                    self._child_cpu[p.pid] = t.user + t.system - self._child_base.get(p.pid, 0.0)
            except psutil.Error:
                pass
        self._peak_rss = max(self._peak_rss, rss)
        return rss

    def _sampler(self):
        while not self._stop.is_set():
            self._poll()
            self._sys_cpu.append(psutil.cpu_percent(interval=None))
            self._stop.wait(self.interval)

    def __enter__(self):
        self._child_cpu, self._sys_cpu, self._peak_rss = {}, [], 0
        self._child_base = {}
        for p in self._procs()[1:]:
            try:
                t = p.cpu_times()
                self._child_base[p.pid] = t.user + t.system
            except psutil.Error:
                pass
        self.tracker = None
        if _CODECARBON:
            try:
                kw = dict(project_name=self.name, output_dir=RESULT_DIR, save_to_file=False,
                          log_level="error", measure_power_secs=1, tracking_mode="process")
                self.tracker = (OfflineEmissionsTracker(country_iso_code=COUNTRY_ISO_CODE, **kw)
                                if COUNTRY_ISO_CODE else EmissionsTracker(**kw))
                self.tracker.start()
            except Exception as e:
                print("[ResourceTracker] codecarbon loi:", e)
                self.tracker = None
        self.base_rss = self._poll()
        self._child_cpu = {}
        psutil.cpu_percent(interval=None)
        t = self.proc.cpu_times()
        self._cpu0 = t.user + t.system
        self._stop.clear()
        self._thread = threading.Thread(target=self._sampler, daemon=True)
        self._thread.start()
        self._t0 = time.perf_counter()
        return self

    def __exit__(self, *exc):
        self.time_sec = time.perf_counter() - self._t0
        t = self.proc.cpu_times()
        main_cpu = t.user + t.system - self._cpu0
        self._stop.set()
        self._thread.join(timeout=2)
        self._poll()
        cpu_sec = main_cpu + sum(self._child_cpu.values())

        self.energy_kwh, self.carbon_kg = float("nan"), float("nan")
        if self.tracker is not None:
            try:
                self.carbon_kg = float(self.tracker.stop() or 0.0)
                data = self.tracker.final_emissions_data
                self.energy_kwh = float(data.energy_consumed) if data is not None else float("nan")
            except Exception as e:
                print("[ResourceTracker] codecarbon stop loi:", e)

        n_cpu = os.cpu_count() or 1
        self.metrics = {
            "time_sec": self.time_sec,
            "cpu_time_sec": cpu_sec,
            # % tren tong nang luc may (100% = dung het n_cpu core)
            "cpu_util_percent": 100.0 * cpu_sec / (self.time_sec * n_cpu) if self.time_sec > 0 else float("nan"),
            "avg_cores_used": cpu_sec / self.time_sec if self.time_sec > 0 else float("nan"),
            "system_cpu_percent_mean": float(np.mean(self._sys_cpu)) if self._sys_cpu else float("nan"),
            "peak_memory_mb": self._peak_rss / 1024 ** 2,
            "peak_memory_increase_mb": (self._peak_rss - self.base_rss) / 1024 ** 2,
            "energy_kwh": self.energy_kwh,
            "energy_joule": self.energy_kwh * 3.6e6,
            "edp_joule_sec": self.energy_kwh * 3.6e6 * self.time_sec,
            "carbon_kg_co2eq": self.carbon_kg,
            "carbon_g_co2eq": self.carbon_kg * 1e3,
        }
        return False

## Thuật toán: CHNMF (Yu et al., IEEE TCYB 2021)

In [ ]:
PREFIX = "CHNMF"        # ten cot dac trung moi: CHNMF1, CHNMF2, ...
N_TOP_GENES = 20     # so gene top |loading| luu cho moi thanh phan
SELECT_INFO = {}   # thong tin phu cua lan fit gan nhat, duoc ghi vao metrics_per_fold.csv

def top_loading_genes(W, genes, comp_names, n_top):
    # W: (n_comp, n_gene) -> top n_top gene co |trong so| lon nhat cua moi thanh phan
    rows = []
    for c, w in zip(comp_names, np.asarray(W)):
        idx = np.argsort(-np.abs(w), kind="stable")[:n_top]
        rows += [{"component": c, "rank": r + 1, "gene": genes[i], "loading": float(w[i])}
                 for r, i in enumerate(idx)]
    return pd.DataFrame(rows)

from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans
from sklearn.metrics import confusion_matrix
from sklearn.metrics.pairwise import euclidean_distances

EPS = 1e-10


def hypergraph_laplacian(Xs, k):
    """Xs: (n mau, m gene). Moi mau sinh 1 sieu canh {v_i} + kNN(v_i). Tra ve (D_v, E) voi L_hyper = D_v - E."""
    n = Xs.shape[0]
    k = min(k, n - 1)
    d2 = euclidean_distances(Xs, squared=True)
    np.fill_diagonal(d2, np.inf)                                  # khong tu chon chinh no lam lang gieng
    knn = np.argsort(d2, axis=1, kind="stable")[:, :k]
    np.fill_diagonal(d2, 0.0)
    nbr = np.hstack([np.arange(n)[:, None], knn])                 # (n, k+1): chinh no + k lang gieng
    H = np.zeros((n, n))                                          # H(v, e_i) (7)
    H[nbr, np.arange(n)[:, None]] = 1.0
    dist = np.take_along_axis(d2, nbr, axis=1)                    # ||v_i - v_j||^2, v_j in e_i
    delta = dist.sum(1) / k                                       # delta = sum ||v_i - v_j||^2 / k
    delta[delta <= 0] = 1.0
    w = np.exp(-dist / delta[:, None]).sum(1)                     # W_i (8)
    f = H.sum(0)                                                  # f(e) = |e| (10)
    E = (H * (w / f)) @ H.T                                       # H W D_e^-1 H^T
    dv = H @ w                                                    # d(v) (9)
    return dv, E


def _row_err(Xg, U, V):
    R = Xg - U @ V.T
    return (R * R).sum(1)                                         # e_i = sum_j (X - UV^T)_ij^2


def chnmf(Xg, k, lam, dv, E, rng):
    """Xg: (m gene, n mau) khong am. Algorithm 1: sigma (16) -> D (20) -> U (26) -> V (27)."""
    m, n = Xg.shape
    scale = np.sqrt(Xg.mean() / k)
    U = rng.uniform(0, 1, (m, k)) * scale + EPS
    V = rng.uniform(0, 1, (n, k)) * scale + EPS
    hist, prev = [], None
    for it in range(1, CHNMF_MAX_ITER + 1):
        e = _row_err(Xg, U, V)
        sigma2 = max(e.sum() / m, EPS)                            # (16)
        d = np.exp(-e / (2 * sigma2))                             # (20): trong so correntropy tung gene
        dU = d[:, None]
        U *= (dU * (Xg @ V)) / (dU * (U @ (V.T @ V))).clip(EPS)  # (26)
        DU = dU * U
        V *= (Xg.T @ DU + lam * (E @ V)) / (V @ (U.T @ DU) + lam * (dv[:, None] * V)).clip(EPS)   # (27)
        e = _row_err(Xg, U, V)
        obj = float((d * e).sum() + lam * (np.einsum("ik,ik->", V, dv[:, None] * V) - np.einsum("ik,ik->", V, E @ V)))
        hist.append(obj)                                          # ham muc tieu (18) voi D hien tai
        if prev is not None and abs(prev - obj) / max(abs(prev), EPS) < CHNMF_TOL:
            break
        prev = obj
    return U, V, d, sigma2, hist


def _nmf_transform(Xg_new, U, d, n_iter, rng):
    # Giai V_new voi U, D co dinh: V <- V * (X^T D U) / (V U^T D U)  (luat (27) bo so hang hypergraph)
    DU = d[:, None] * U
    G = U.T @ DU
    num = Xg_new.T @ DU
    V = rng.uniform(0, 1, (Xg_new.shape[1], U.shape[1])) * np.sqrt(Xg_new.mean() / U.shape[1]) + EPS
    for _ in range(n_iter):
        V *= num / (V @ G).clip(EPS)
    return V


def clustering_ac(V, y):
    # AC (32): K-means tren V, anh xa cum -> nhan bang Kuhn-Munkres
    c = len(np.unique(y))
    pred = KMeans(n_clusters=c, n_init=10, random_state=SEED).fit_predict(V)
    cm = confusion_matrix(y, pred)
    r, col = linear_sum_assignment(-cm)
    return cm[r, col].sum() / len(y)


class CHNMFModel:
    def __init__(self, U, d, shift, order):
        self.U, self.d, self.shift, self.order = U, d, shift, order

    def prep(self, X):
        return np.clip(X + self.shift, 0, None).T                # (m gene, n mau)


def resolve_k(X, y):
    return min(N_FEATURES, X.shape[0], X.shape[1])


def fit_extractor(X, y, k):
    SELECT_INFO.clear()
    shift = max(0.0, -float(X.min()))     # tinh tien neu train co gia tri am
    Xg = np.clip(X + shift, 0, None).T     # (m gene, n mau) nhu bai bao
    dv, E = hypergraph_laplacian(Xg.T, CHNMF_HG_K)

    lam, grid_ac = CHNMF_LAMBDA, {}
    if CHNMF_LAMBDA_GRID:
        for l in CHNMF_LAMBDA_GRID:
            _, Vl, *_ = chnmf(Xg, k, l, dv, E, np.random.default_rng(SEED))
            grid_ac[l] = float(clustering_ac(Vl, y))
        lam = max(grid_ac, key=grid_ac.get)
        print("  [chnmf] AC phan cum theo lambda:", {f"{l:g}": round(a, 3) for l, a in grid_ac.items()}, "-> chon", lam)

    U, V, d, sigma2, hist = chnmf(Xg, k, lam, dv, E, np.random.default_rng(SEED))

    # Dong gop cua thanh phan j ~ ||u_j|| * ||v_j|| -> xep giam dan
    contrib = np.linalg.norm(U, axis=0) * np.linalg.norm(V, axis=0)
    contrib = contrib / contrib.sum() if contrib.sum() > 0 else contrib
    order = np.argsort(-contrib, kind="stable")
    model = CHNMFModel(U, d, shift, order)
    Un = U / np.linalg.norm(U, axis=0).clip(EPS)                 # chuan hoa cot U truoc khi cham diem gene
    model.gene_scores = Un.sum(1)
    model.history = hist
    SELECT_INFO.update(lambda_used=lam, n_iter=len(hist), converged=len(hist) < CHNMF_MAX_ITER,
                       final_objective=hist[-1], sigma2=float(sigma2), shift=shift,
                       recon_mse_train=float(_row_err(Xg, U, V).sum() / Xg.size),
                       correntropy_weight_min=float(d.min()), correntropy_weight_mean=float(d.mean()),
                       train_clustering_ac=float(clustering_ac(V, y)),
                       lambda_grid_ac=json.dumps({f"{l:g}": a for l, a in grid_ac.items()}) if grid_ac else None)
    return model, V[:, order], contrib[order]


def transform(model, X):
    V = _nmf_transform(model.prep(X), model.U, model.d, CHNMF_TRANSFORM_ITER, np.random.default_rng(SEED))
    return V[:, model.order]


def loadings(model):
    return model.U[:, model.order].T                              # (k, m gene)

## Chạy trên 5 fold

In [ ]:
all_rows, metric_rows, loading_rows = [], [], []
VIS = {}   # luu fold 1 de ve hinh
HISTORY = {}   # duong hoi tu ham muc tieu cua tung fold

for fold in range(1, N_FOLDS + 1):
    X, y, genes, ids = load_split(fold, "train")   # CHI dung tap train de fit
    Xv, yv, genes_v, ids_v = load_split(fold, "val")
    Xt, yt, genes_t, ids_t = load_split(fold, "test")
    assert (genes_v == genes).all() and (genes_t == genes).all(), "Thu tu gene train/val/test khac nhau"
    k = resolve_k(X, y)

    with ResourceTracker(f"{METHOD}_{DATASET_NAME}_fold{fold}") as rt:
        model, Z, scores = fit_extractor(X, y, k)

    t0 = time.perf_counter()
    Zv, Zt = transform(model, Xv), transform(model, Xt)
    transform_time = time.perf_counter() - t0

    names = [f"{PREFIX}{i}" for i in range(1, Z.shape[1] + 1)]
    out_dir = os.path.join(RESULT_DIR, "features", f"fold_{fold}")
    os.makedirs(out_dir, exist_ok=True)
    for split, Zs, ys, idx_s in [("train", Z, y, ids), ("val", Zv, yv, ids_v), ("test", Zt, yt, ids_t)]:
        df = pd.DataFrame(np.asarray(Zs, dtype=np.float64), index=idx_s, columns=names)
        df["label"] = ys
        df.to_hdf(os.path.join(out_dir, f"{split}.h5"), key="data", mode="w")

    comp = pd.DataFrame({"fold": fold, "rank": np.arange(1, len(names) + 1), "component": names, "score": scores})
    comp.to_csv(os.path.join(RESULT_DIR, "components", f"fold_{fold}.csv"), index=False)
    all_rows.append(comp)
    W = loadings(model)
    if W is not None:
        loading_rows.append(top_loading_genes(W, genes, names, N_TOP_GENES).assign(fold=fold))
    if getattr(model, "gene_scores", None) is not None:
        # Xep hang gene theo diem tu ma tran co so U (phan feature selection cua bai bao)
        os.makedirs(os.path.join(RESULT_DIR, "gene_scores"), exist_ok=True)
        gs = np.asarray(model.gene_scores)
        o = np.argsort(-gs, kind="stable")[:N_TOP_SCORED_GENES]
        pd.DataFrame({"fold": fold, "rank": np.arange(1, len(o) + 1), "gene": genes[o], "score": gs[o]}) \
            .to_csv(os.path.join(RESULT_DIR, "gene_scores", f"fold_{fold}.csv"), index=False)
    if getattr(model, "history", None) is not None:
        HISTORY[fold] = model.history

    m = {"dataset": DATASET_NAME, "method": METHOD, "fold": fold,
         "n_train": X.shape[0], "n_val": Xv.shape[0], "n_test": Xt.shape[0],
         "n_genes_total": X.shape[1], "n_components": Z.shape[1],
         "transform_time_sec": transform_time, **SELECT_INFO, **rt.metrics}
    metric_rows.append(m)
    print(f"fold {fold}: {X.shape} -> {Z.shape[1]} chieu | {m['time_sec']:.2f}s | CPU {m['cpu_util_percent']:.1f}% "
          f"({m['avg_cores_used']:.2f} core) | peak {m['peak_memory_mb']:.0f} MB | "
          f"{m['energy_kwh']:.3e} kWh | {m['carbon_g_co2eq']:.4f} gCO2eq | transform {transform_time:.2f}s")
    if fold == 1:
        VIS = {"train": (np.asarray(Z), y), "test": (np.asarray(Zt), yt), "names": names}
    del X, Xv, Xt, model

components = pd.concat(all_rows, ignore_index=True)
components.to_csv(os.path.join(RESULT_DIR, "components_all_folds.csv"), index=False)
if loading_rows:
    pd.concat(loading_rows, ignore_index=True).to_csv(os.path.join(RESULT_DIR, "loadings_top_genes.csv"), index=False)

metrics = pd.DataFrame(metric_rows)
metrics.to_csv(os.path.join(RESULT_DIR, "metrics_per_fold.csv"), index=False)
metrics

## Tổng hợp độ đo

In [ ]:
cols = ["time_sec", "cpu_time_sec", "cpu_util_percent", "avg_cores_used", "system_cpu_percent_mean",
        "peak_memory_mb", "peak_memory_increase_mb", "energy_kwh", "energy_joule",
        "edp_joule_sec", "carbon_kg_co2eq", "carbon_g_co2eq", "transform_time_sec"]
summary = metrics[cols].agg(["mean", "std", "min", "max"]).T
summary.loc["time_sec", "total"] = metrics["time_sec"].sum()
summary.loc["energy_kwh", "total"] = metrics["energy_kwh"].sum()
summary.loc["carbon_kg_co2eq", "total"] = metrics["carbon_kg_co2eq"].sum()
summary.to_csv(os.path.join(RESULT_DIR, "metrics_summary.csv"))

with open(os.path.join(RESULT_DIR, "config.json"), "w") as f:
    json.dump({"dataset": DATASET_NAME, "data_dir": DATA_DIR, "method": METHOD, "n_features": N_FEATURES,
               "n_folds": N_FOLDS, "seed": SEED, "params": METHOD_PARAMS, "cpu_count": os.cpu_count(),
               "country_iso_code": COUNTRY_ISO_CODE}, f, indent=1, default=str)
summary

## Trực quan hoá không gian mới (tham khảo)

In [ ]:
# Tham khao nhanh: 2 chieu dau tien cua fold 1 (train: cham tron, test: dau x), to mau theo stage
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6.5, 5))
names = VIS["names"]
if len(names) >= 2:
    for split, marker, alpha in [("train", "o", 0.45), ("test", "x", 0.9)]:
        Zs, ys = VIS[split]
        for c in np.unique(ys):
            mk = ys == c
            ax.scatter(Zs[mk, 0], Zs[mk, 1], marker=marker, s=18, alpha=alpha,
                       color=plt.cm.tab10(int(c)), label=f"{STAGE_NAMES.get(int(c), c)} ({split})")
    ax.set_xlabel(names[0]); ax.set_ylabel(names[1])
else:   # chi co 1 chieu (vd LDA 2 lop) -> ve phan bo 1 chieu
    for split, marker in [("train", "o"), ("test", "x")]:
        Zs, ys = VIS[split]
        ax.scatter(Zs[:, 0], ys + (0.15 if split == "test" else 0), marker=marker, c=ys, cmap="tab10", s=18)
    ax.set_xlabel(names[0]); ax.set_ylabel("label")
ax.set_title(f"{METHOD} | {DATASET_NAME} | fold 1")
ax.legend(fontsize=7, ncol=2, loc="best")
fig.tight_layout()
fig.savefig(os.path.join(RESULT_DIR, "scatter_fold1.png"), dpi=150)
plt.show()

## Hội tụ của CHNMF

In [ ]:
# Duong hoi tu ham muc tieu (18) qua cac vong lap (tuong tu Fig. 3 cua bai bao)
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6.5, 4))
for fold, h in HISTORY.items():
    ax.plot(np.arange(1, len(h) + 1), h, label=f"fold {fold}")
ax.set_xlabel("vong lap"); ax.set_ylabel("ham muc tieu (18)"); ax.set_yscale("log")
ax.set_title(f"CHNMF hoi tu | {DATASET_NAME}")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(RESULT_DIR, "convergence.png"), dpi=150)
plt.show()

In [ ]:
import shutil
zip_path = shutil.make_archive(f"/kaggle/working/fe_{METHOD}_{DATASET_NAME}", "zip", RESULT_DIR)
print("Da nen:", zip_path)
for root, _, files in os.walk(RESULT_DIR):
    for fn in sorted(files):
        print(" ", os.path.relpath(os.path.join(root, fn), RESULT_DIR))